# iNaturalist batch metadata extraction

Loop over all CSV files under `data/experiment/inaturalist` (or the bundled `inaturalist_100_species_1000_obs` fallback), run the full pipeline with the `croissant_inaturalist_standard`, and save JSON outputs under `outputs/inat_<model_slug>/`.

Set `LLM_MODEL` (and provider) in `.env` before running.

In [1]:
import json
import logging
import os
import re
import sys
from pathlib import Path

BASE = Path('../..').resolve()
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv

# Load credentials/defaults, then force this notebook run to one model/provider.
load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'
os.environ['LLM_MODEL'] = 'Sehyo/Qwen3.5-122B-A10B-NVFP4'

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import run_metadata_extraction
from src.standards import METADATA_STANDARDS

STANDARD_NAME = 'croissant_standard_subset'
TOPOLOGY = 'single'

LLM_NAME = get_model_name()
LLM_SLUG = re.sub(r'[^\w.\-]+', '_', LLM_NAME.replace('/', '_'))
EXPERIMENT_RUN = f'cybench_{LLM_SLUG}'

INATURALIST_DIR_CANDIDATES = [
    # Default cybench location (preferred when DATA_DIR is not set).
    BASE / 'data/experiment/cybench-maize-IE/IE',
    # Default iNaturalist locations (legacy behavior).
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir():
            return path
    searched = '\n  '.join(str(p) for p in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


def collect_inaturalist_csv_files(data_dir: Path) -> list[Path]:
    return sorted(data_dir.glob('*.csv'))


DATA_DIR_OVERRIDE = os.environ.get('DATA_DIR')
DATASET_TAG = os.environ.get('DATASET_TAG')
CONTEXT_PREFIX = os.environ.get('CONTEXT_PREFIX')

if DATA_DIR_OVERRIDE:
    INATURALIST_DIR = Path(DATA_DIR_OVERRIDE).resolve()
else:
    INATURALIST_DIR = resolve_inaturalist_dir()

# When overriding the input directory (e.g. cybench batches), put outputs
# in a dataset-tag subfolder to avoid collisions.
if DATA_DIR_OVERRIDE:
    if not DATASET_TAG:
        DATASET_TAG = INATURALIST_DIR.name
    OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN / DATASET_TAG
else:
    # If the resolved default input is cybench, also write into a dataset-tag subfolder.
    if 'cybench-maize-IE' in str(INATURALIST_DIR):
        if not DATASET_TAG:
            DATASET_TAG = INATURALIST_DIR.name
        OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN / DATASET_TAG
    else:
        OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN

if not CONTEXT_PREFIX:
    if DATA_DIR_OVERRIDE:
        CONTEXT_PREFIX = f'cybench_{DATASET_TAG}'
    else:
        # Infer the prefix from the resolved default input directory.
        CONTEXT_PREFIX = f'cybench_{INATURALIST_DIR.name}' if 'cybench-maize-IE' in str(INATURALIST_DIR) else 'inaturalist'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

metadata_standard = METADATA_STANDARDS[STANDARD_NAME]


def output_path_for(csv_path: Path) -> Path:
    return OUTPUT_DIR / f'{csv_path.stem}_inaturalist.json'


all_csv_files = collect_inaturalist_csv_files(INATURALIST_DIR)
if not all_csv_files:
    raise FileNotFoundError(f'No CSV files found in {INATURALIST_DIR}')

csv_files = [p for p in all_csv_files if not output_path_for(p).exists()]
skipped_count = len(all_csv_files) - len(csv_files)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'LLM model: {LLM_NAME}')
print(f'Experiment run: {EXPERIMENT_RUN}')
print(f'Data directory: {INATURALIST_DIR}')
print(f'Found {len(all_csv_files)} CSV file(s)')
print(f'Skipping {skipped_count} already completed in {OUTPUT_DIR}')
print(f'Remaining to run: {len(csv_files)}')
print(f'Output directory: {OUTPUT_DIR}')

LLM provider: surf
LLM model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Experiment run: cybench_Sehyo_Qwen3.5-122B-A10B-NVFP4
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/cybench-maize-IE/IE
Found 9 CSV file(s)
Skipping 0 already completed in /home/com3dian/Github/metadata_agent/outputs/cybench_Sehyo_Qwen3.5-122B-A10B-NVFP4/IE
Remaining to run: 9
Output directory: /home/com3dian/Github/metadata_agent/outputs/cybench_Sehyo_Qwen3.5-122B-A10B-NVFP4/IE


In [2]:
SPATIAL_KEYS = ('min_lat', 'min_lon', 'max_lat', 'max_lon')
TEMPORAL_KEYS = ('from', 'to')


def normalize_inaturalist_metadata(metadata):
    """Keep only schema-shaped iNaturalist Croissant metadata."""
    if not isinstance(metadata, dict):
        return None
    if not {'spatialCoverage', 'temporalCoverage'}.issubset(metadata.keys()):
        return None

    spatial = metadata.get('spatialCoverage')
    temporal = metadata.get('temporalCoverage')

    if spatial is not None:
        if not isinstance(spatial, dict) or not set(SPATIAL_KEYS).issubset(spatial.keys()):
            return None
        spatial = {key: spatial[key] for key in SPATIAL_KEYS}

    if temporal is not None:
        if not isinstance(temporal, dict) or not set(TEMPORAL_KEYS).issubset(temporal.keys()):
            return None
        temporal = {key: temporal[key] for key in TEMPORAL_KEYS}

    if spatial is None and temporal is None:
        return None

    return {
        'spatialCoverage': spatial,
        'temporalCoverage': temporal,
    }


def metadata_from_step_results(metadata_result):
    """Fallback when synthesis leaves metadata_output empty.

    For croissant_standard_subset we keep the full metadata JSON as-is.
    """
    for step in reversed(metadata_result.step_results or []):
        for player_result in step.individual_results or []:
            analysis = player_result.get('analysis')
            if not analysis:
                continue

            candidates = [analysis]
            if isinstance(analysis, str):
                candidates = [analysis.strip()]
                start = analysis.find('{')
                end = analysis.rfind('}')
                if start != -1 and end > start:
                    candidates.append(analysis[start:end + 1])

            for candidate in candidates:
                if isinstance(candidate, dict):
                    return candidate
                elif isinstance(candidate, str) and candidate:
                    try:
                        parsed = json.loads(candidate)
                    except json.JSONDecodeError:
                        parsed = None
                    if isinstance(parsed, dict):
                        return parsed

    return None


def extract_metadata(metadata_result):
    if metadata_result is None:
        return None

    # Keep the full croissant_standard_subset JSON (no spatial/temporal filtering).
    if getattr(metadata_result, 'final_metadata', None) is not None:
        return metadata_result.final_metadata

    workspace = metadata_result.final_workspace or {}
    if workspace.get('metadata_output') is not None:
        return workspace.get('metadata_output')

    return metadata_from_step_results(metadata_result)


def is_empty_metadata(metadata):
    return metadata is None or (isinstance(metadata, dict) and not metadata)


def run_pipeline(csv_path: Path, attempt: int = 1):
    context_name = f'{CONTEXT_PREFIX}_{csv_path.stem}'
    print(f'\n{"=" * 60}')
    print(f'[{attempt}] Processing {csv_path.name} (context: {context_name})')

    return run_metadata_extraction(
        source=str(csv_path.resolve()),
        metadata_standard=metadata_standard,
        metadata_standard_name=STANDARD_NAME,
        name=context_name,
        topology_name=TOPOLOGY,
    )


results_summary = []
success_count = 0
skipped_count = 0

# Run all CSVs in this folder as ONE multi_csv dataset.
# The existing per-CSV loop below will be skipped by emptying `csv_files`.
dataset_context_name = f'{CONTEXT_PREFIX}_multi_csv'
dataset_output_file = OUTPUT_DIR / f'{dataset_context_name}.json'

csv_files = []

if dataset_output_file.exists():
    print(f'Skipping multi-csv dataset; output exists: {dataset_output_file}')
    skipped_count = 1
else:
    last_exc = None
    final_attempt = None
    metadata = None
    result = None
    for attempt in (1, 2):
        final_attempt = attempt
        try:
            print(f'\n[attempt {attempt}] Running multi_csv extraction with {len(all_csv_files)} CSV files')
            result = run_metadata_extraction(
                source=[str(p.resolve()) for p in all_csv_files],
                metadata_standard=metadata_standard,
                metadata_standard_name=STANDARD_NAME,
                name=dataset_context_name,
                topology_name=TOPOLOGY,
            )
            metadata = extract_metadata(result)
        except Exception as exc:
            last_exc = str(exc)
            print(f'  Attempt {attempt} failed: {exc}')
            continue

        if not is_empty_metadata(metadata):
            break

    if is_empty_metadata(metadata):
        print(f'  FAILED: empty metadata for multi_csv dataset: {dataset_context_name}')
        results_summary.append({
            'csv': '<multi_csv>',
            'success': False,
            'output_file': None,
            'attempts': final_attempt or 2,
            'error': last_exc or 'empty metadata',
        })
    else:
        with open(dataset_output_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)

        print(f'Saved: {dataset_output_file}')
        results_summary.append({
            'csv': '<multi_csv>',
            'success': True,
            'output_file': str(dataset_output_file),
            'attempts': final_attempt or 1,
            'error': None,
        })
        success_count = 1
        skipped_count = 0

for csv_path in csv_files:
    entry = {
        'csv': csv_path.name,
        'success': False,
        'output_file': None,
        'attempts': 0,
        'error': None,
    }

    metadata = None
    for attempt in (1, 2):
        entry['attempts'] = attempt
        try:
            result = run_pipeline(csv_path, attempt=attempt)
            metadata = extract_metadata(result)
        except Exception as exc:
            entry['error'] = str(exc)
            print(f'  Attempt {attempt} failed: {exc}')
            continue

        if not is_empty_metadata(metadata):
            break

        print(f'  Attempt {attempt} returned empty metadata' + (' — retrying' if attempt == 1 else ''))

    if is_empty_metadata(metadata):
        print(f'  FAILED: no metadata for {csv_path.name}')
        results_summary.append(entry)
        continue

    output_file = output_path_for(csv_path)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    entry['success'] = True
    entry['output_file'] = str(output_file)
    success_count += 1
    print(f'  Saved: {output_file}')
    results_summary.append(entry)

print(f'\n{"=" * 60}')
print(f'Batch complete: {success_count}/1 successful this run')
print(f'Previously completed (skipped): {skipped_count}')
print(f'Total with outputs: {skipped_count + success_count}/1')
for item in results_summary:
    status = 'OK' if item['success'] else 'FAIL'
    detail = item['output_file'] or item.get('error') or 'empty metadata'
    print(f'  [{status}] {item["csv"]} ({item["attempts"]} attempt(s)) -> {detail}')


[attempt 1] Running multi_csv extraction with 9 CSV files


/home/com3dian/Github/metadata_agent/src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-08 17:22:39,242 - INFO - root - PlanExecutor initialized with topology: single
2026-07-08 17:22:39,243 - INFO - root -   Players per step: 1
2026-07-08 17:22:39,243 - INFO - root -   Debate rounds: 0
2026-07-08 17:22:39,244 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-08 17:22:39,244 - INFO - root - Orchestrator initialized with topology: single
2026-07-08 17:22:39,245 - INFO - root - ============================================================
2026-07-08 17:22:39,245 - INFO - root - STARTING ORCHESTRATION
2026-07-08 17:22:39,246 - INFO - root - Context: cybench_IE_multi_csv
2026-07-08 17:22:39,246 - INFO - root - Type: multi_csv
2026-07-08 17:22:39,246 - INFO - root - Resources: ['crop_calendar_maize_IE', 'crop_mask_maize_IE', 'fpar_maize_IE', 'location_maize_IE', 'meteo_maize_IE', 'ndvi_maize_IE', 'soil_maize_IE', 'soil_moisture_maize_I

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-08 17:22:49,159 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-08 17:22:49,181 - WARNING - root - Plan validation warning: Step 3 ('Generate Croissant metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-08 17:22:49,181 - INFO - root - Plan generated successfully!
2026-07-08 17:22:49,182 - INFO - root - Number of steps: 4
2026-07-08 17:22:49,182 - INFO - root -   Step 1: Preview schema and first rows for all resources (player: dataset_schema_preview)
2026-07-08 17:22:49,182 - INFO - root -   Step 2: Discover relationships between resources (player: relationship_analyst)
2026-07-08 17:22:49,182 - INFO - root -   Step 3: Detect and analyze spatial and temporal extents (player: spatial_temporal_specialist) (resources: ['location_maize_IE', 'fpar_maize_IE', 'meteo_maize_IE', 'ndvi_maize_IE', 'soil_moisture_maize_IE', 'yield_maize_IE'])
2026-07-08 17:22:49,182 - INFO - root -   Step 4: Gener

...

2026-07-08 17:22:50,292 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-08 17:22:50,308 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop iteration 1 — model requested 1 tool(s): get_columns_with_first_row
2026-07-08 17:23:10,462 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-08 17:23:10,463 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop finished after 2 iteration(s), 1 tool invocation(s) — model returned final analysis
2026-07-08 17:23:10,464 - INFO - root - Player 'dataset_schema_preview_1': using LLM tool path — 1 tool result(s), 1 trace entry/entries
2026-07-08 17:23:10,464 - INFO - root -   Player 'dataset_schema_preview_1' completed execution
2026-07-08 17:23:10,464 - INFO - root -     Output: ### Analysis of Dataset Schema and First Rows  #### Approach To fulfill the request of previewing the schema and first rows for all res

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-08 17:23:39,503 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-08 17:23:39,504 - INFO - root - Player 'relationship_analyst_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-07-08 17:23:39,505 - INFO - root - Player 'relationship_analyst_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-07-08 17:23:39,505 - INFO - root -   Player 'relationship_analyst_1' completed execution
2026-07-08 17:23:39,505 - INFO - root -     Output: I'll analyze the relationships between all resources in the multi_csv context. Let me start by getting an overview of the context and then examine the relationships.  
2026-07-08 17:23:39,506 - INFO - root - Single player, skipping debate
2026-07-08 17:23:39,507 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-07-08 17:23:41,081 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 2

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-08 17:26:43,684 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-08 17:26:43,692 - INFO - root - Player 'croissant_metadata_generator_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-07-08 17:26:43,692 - INFO - root - Player 'croissant_metadata_generator_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-07-08 17:26:43,692 - INFO - root -   Player 'croissant_metadata_generator_1' completed execution
2026-07-08 17:26:43,692 - INFO - root -     Output: ```json {   "name": "Maize Agricultural and Environmental Data for Ireland",   "description": "A comprehensive dataset containing maize crop calendar, area, meteorological, vegetation index, soil, and...
2026-07-08 17:26:43,693 - INFO - root - Single player, skipping debate
2026-07-08 17:26:43,693 - INFO - root - --- STEP 3: SYNTHESIS ---
2026-07-08 17:26:43,693 - INFO - root -   Using structured